# 0단계: 평가셋 만들기 + v3 Before 측정

이 노트북은 **두 번에 나눠서** 실행합니다.

| 파트 | 언제 | 하는 일 |
|---|---|---|
| **A. 사전 라벨 만들기** | 지금 | 0144, 0156의 득점 플레이 구간에서 프레임 10장씩 추출 → 박스를 미리 채운 라벨 생성 → Roboflow 업로드용 zip |
| *(Roboflow에서 라벨 수정)* | 1시간 남짓 | 틀린 박스 삭제, 빠진 선수 추가, 클래스 수정 |
| **B. 평가** | 라벨링 후 | 정답 라벨로 v3 성능 측정 → `eval_results.csv`에 **v3_baseline** 기록 |

이후 단계(탐지/추적/분류 개선)에서 새 JSON이 나오면 **파트 B의 `evaluate_run()`만 다시 호출**하면 같은 정답으로 채점됩니다.

**실행 전:** 런타임 → 런타임 유형 변경 → **T4 GPU** (파트 A의 고해상도 탐지에 사용)

## 공통 설정

In [ ]:
!pip install -q ultralytics
import os, json, glob, shutil, zipfile, re
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from scipy.optimize import linear_sum_assignment
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ===== 경로 (v3 노트북 Cell 2와 동일하게) =====
FOLDER_PATH      = "/content/drive/MyDrive/2025년 홍익 vs 국민 드론뷰"
DRIVE_OUTPUT_DIR = os.path.join(FOLDER_PATH, "pipeline_outputs")   # v3의 tracking_final.json 위치
EVAL_DIR         = os.path.join(DRIVE_OUTPUT_DIR, "eval")           # 이번에 새로 만드는 평가 폴더
IMG_DIR          = os.path.join(EVAL_DIR, "images")                 # 평가용 프레임 20장 (정답 라벨의 기준 이미지)
PRE_DIR          = os.path.join(EVAL_DIR, "prelabels")              # 사전 라벨
for _d in (EVAL_DIR, IMG_DIR, PRE_DIR):
    os.makedirs(_d, exist_ok=True)

# ===== 클래스 =====
# 정답(GT)에는 0~2만 남아야 합니다. 3(Unknown)은 "라벨러가 확인해야 할 박스" 표시용입니다.
CLASS_NAMES = ["Hongik", "Kookmin", "Referee", "Unknown"]
CLS_ID = {n: i for i, n in enumerate(CLASS_NAMES)}

# ===== 평가 클립 =====
# play_range : 득점 플레이 구간 (v3 frame_id 기준, 1부터 시작, 양 끝 포함)
# scorer     : v3에서 득점 선수 경로를 만들 때 쓴 (track_id, 프레임 구간). 득점 선수 '정답 위치'로 사용.
#              ⚠️ 0156의 #171, #183 구간은 v3 노트북 주석에 '추정, 확인 필요'로 되어 있었습니다.
#                 v3의 프레임 브라우저 셀로 확인 후 틀렸다면 여기 숫자를 고치세요.
EVAL_CLIPS = {
    "DJI_20250913133612_0144_D": {
        "clip_key": 34,
        "play_range": (413, 649),
        "scorer": [(2122, (413, 649))],
    },
    "DJI_20250913135020_0156_D": {
        "clip_key": 44,
        "play_range": (167, 525),
        "scorer": [(184, (167, 214)), (171, (215, 313)), (183, (314, 393)), (208, (410, 525))],
    },
}
FRAMES_PER_CLIP = 10

# ===== 매칭 기준 =====
# 선수가 10~25px로 작아서 IoU 0.5는 박스가 조금만 어긋나도 오답이 됩니다.
# 대신 '박스 중심 거리'로 짝을 짓습니다: 거리 <= max(MIN_PX, RATIO × 정답 박스 긴 변)
MATCH_MIN_PX = 10
MATCH_RATIO  = 0.5
SCORER_MATCH_PX = 15   # 득점 선수 정답 위치(발)와 예측 박스 발 위치가 이 거리(px) 이내면 '찾았다'


def video_path_of(stem):
    for ext in (".MP4", ".mp4", ".MOV", ".mov"):
        p = os.path.join(FOLDER_PATH, stem + ext)
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"영상을 찾을 수 없습니다: {stem}")

def tracking_json_of(stem, folder=DRIVE_OUTPUT_DIR):
    return os.path.join(folder, f"{stem}_tracking_final.json")

for stem in EVAL_CLIPS:
    print(stem, "| 영상:", os.path.exists(video_path_of(stem)), "| v3 JSON:", os.path.exists(tracking_json_of(stem)))

In [ ]:
# ===== 공통 함수 =====

def load_tracking(json_path):
    """v3 형식 JSON → {frame_id: [player dict, ...]}, field_polygon"""
    with open(json_path, encoding="utf-8") as f:
        data = json.load(f)
    by_frame = {fr["frame_id"]: fr["players"] for fr in data["frames"]}
    poly = np.array(data.get("field_polygon", []), dtype=np.int32)
    return by_frame, poly


def read_frames_sequential(video_path, frame_ids):
    """
    frame_id(1부터 시작)에 해당하는 프레임을 '처음부터 순서대로 읽어서' 가져옵니다.
    cap.set(POS_FRAMES)로 바로 점프하면 드론 영상(H.264/H.265)에서 몇 프레임씩 어긋날 수 있어서,
    v3가 프레임을 센 방식(처음부터 1, 2, 3...)과 똑같이 맞추기 위해 순차로 읽습니다.
    """
    wanted = set(frame_ids)
    last = max(wanted)
    out = {}
    cap = cv2.VideoCapture(video_path)
    fid = 0
    while fid < last:
        ok, frame = cap.read()
        if not ok:
            break
        fid += 1
        if fid in wanted:
            out[fid] = frame
    cap.release()
    missing = wanted - set(out)
    if missing:
        print(f"⚠️ 읽지 못한 프레임: {sorted(missing)}")
    return out


def pick_frames(play_range, n):
    a, b = play_range
    return sorted(set(np.linspace(a, b, n).round().astype(int).tolist()))


def inside(poly, x, y):
    if poly is None or len(poly) < 3:
        return True
    return cv2.pointPolygonTest(poly.reshape(-1, 1, 2), (float(x), float(y)), False) >= 0


def xyxy_to_yolo_line(cls_id, box, W, H):
    x1, y1, x2, y2 = box
    x1, x2 = max(0, x1), min(W, x2)
    y1, y2 = max(0, y1), min(H, y2)
    cx, cy = (x1 + x2) / 2 / W, (y1 + y2) / 2 / H
    w, h = (x2 - x1) / W, (y2 - y1) / H
    return f"{cls_id} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}"


def yolo_line_to_xyxy(line, W, H):
    c, cx, cy, w, h = line.split()[:5]
    cx, cy, w, h = float(cx) * W, float(cy) * H, float(w) * W, float(h) * H
    return int(c), [cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2]


def center(b):
    return ((b[0] + b[2]) / 2, (b[1] + b[3]) / 2)


def box_iou(a, b):
    """Day 2에서 만든 IoU (xyxy)"""
    iw = max(0.0, min(a[2], b[2]) - max(a[0], b[0]))
    ih = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = iw * ih
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / union if union > 0 else 0.0


def match_by_center(gt_boxes, pred_boxes):
    """
    정답 박스와 예측 박스를 1:1로 짝짓기 (헝가리안 알고리즘, 중심 거리 기준).
    return: [(gi, pi), ...], 짝 없는 정답 인덱스, 짝 없는 예측 인덱스
    """
    G, P = len(gt_boxes), len(pred_boxes)
    if G == 0 or P == 0:
        return [], list(range(G)), list(range(P))
    gc = np.array([center(b) for b in gt_boxes])
    pc = np.array([center(b) for b in pred_boxes])
    dist = np.linalg.norm(gc[:, None, :] - pc[None, :, :], axis=2)          # (G, P)
    gsize = np.array([max(b[2] - b[0], b[3] - b[1]) for b in gt_boxes])
    thr = np.maximum(MATCH_MIN_PX, MATCH_RATIO * gsize)[:, None]            # 정답 박스별 허용 거리
    cost = np.where(dist <= thr, dist, 1e6)
    gi, pi = linear_sum_assignment(cost)
    pairs = [(g, p) for g, p in zip(gi, pi) if cost[g, p] < 1e6]
    mg = {g for g, _ in pairs}
    mp = {p for _, p in pairs}
    return pairs, [g for g in range(G) if g not in mg], [p for p in range(P) if p not in mp]


print("✅ 공통 함수 준비 완료")

---
# 파트 A. 사전 라벨 만들기

박스를 미리 채워두는 방법은 두 가지를 합칩니다.
1. **v3 결과(JSON)**: 박스 + 클래스(Hongik/Kookmin/Referee/Unknown)
2. **고해상도 탐지**(yolov8x, `imgsz=1920`, `conf=0.1`): v3가 놓친 선수까지 최대한 잡기 → 클래스는 `Unknown`

라벨링에서 **"없는 박스 추가"가 가장 오래 걸리고, "틀린 박스 삭제"는 빠릅니다.** 그래서 일부러 박스를 넉넉하게 채웁니다.

In [ ]:
from ultralytics import YOLO
PRELABEL_MODEL  = YOLO("yolov8x.pt")
PRELABEL_IMGSZ  = 1920
PRELABEL_CONF   = 0.10
MERGE_IOU       = 0.3      # v3 박스와 1:1로 짝지어진 고해상도 박스의 IoU가 이보다 크면 같은 사람 → v3 것만 남김
                           # (단순 거리 기준을 쓰면 붙어 있는 다른 선수까지 지워질 수 있어서 1:1 짝짓기를 씀)



def build_prelabels_for_clip(stem, cfg):
    by_frame, poly = load_tracking(tracking_json_of(stem))
    fids = pick_frames(cfg["play_range"], FRAMES_PER_CLIP)
    frames = read_frames_sequential(video_path_of(stem), fids)
    stats = Counter()
    for fid, img in frames.items():
        H, W = img.shape[:2]
        name = f"{stem}_f{fid:05d}"
        cv2.imwrite(os.path.join(IMG_DIR, name + ".jpg"), img, [cv2.IMWRITE_JPEG_QUALITY, 95])

        # 1) v3 박스
        boxes = [(CLS_ID.get(p["type"], CLS_ID["Unknown"]), p["bbox"]) for p in by_frame.get(fid, [])]
        stats["v3"] += len(boxes)

        # 2) 고해상도 탐지 → ROI(발 위치) 안 + v3와 겹치지 않는 것만 Unknown으로 추가
        r = PRELABEL_MODEL.predict(img, imgsz=PRELABEL_IMGSZ, conf=PRELABEL_CONF,
                                   classes=[0], verbose=False)[0]
        v3_boxes = [b for _, b in boxes]
        hi = [b for b in r.boxes.xyxy.cpu().numpy().tolist() if inside(poly, (b[0] + b[2]) / 2, b[3])]
        # 1:1 짝짓기: v3 박스 하나는 고해상도 박스 '하나'만 흡수 → 겹쳐 있는 두 선수 중 v3가 놓친 쪽은 살아남음
        matched = set()
        if v3_boxes and hi:
            iou_m = np.array([[box_iou(h, v) for v in v3_boxes] for h in hi])
            hi_idx, v3_idx = linear_sum_assignment(-iou_m)
            matched = {h for h, v in zip(hi_idx, v3_idx) if iou_m[h, v] > MERGE_IOU}
        for h, b in enumerate(hi):
            if h in matched:
                continue
            boxes.append((CLS_ID["Unknown"], b))
            stats["added_hi_res"] += 1

        with open(os.path.join(PRE_DIR, name + ".txt"), "w") as f:
            f.write("\n".join(xyxy_to_yolo_line(c, b, W, H) for c, b in boxes))
    print(f"{stem}: 프레임 {len(frames)}장 | v3 박스 {stats['v3']}개 + 고해상도 추가 {stats['added_hi_res']}개")


for stem, cfg in EVAL_CLIPS.items():
    build_prelabels_for_clip(stem, cfg)

In [ ]:
# 사전 라벨 미리보기 (v3와 같은 색: 하늘색=Hongik, 주황=Kookmin / 분홍=Referee, 회색=Unknown(확인 필요))
COLORS = {0: (255, 215, 0), 1: (0, 140, 255), 2: (255, 0, 255), 3: (160, 160, 160)}   # OpenCV는 BGR 순서

def draw_label_file(img, txt_path):
    H, W = img.shape[:2]
    out = img.copy()
    if os.path.exists(txt_path):
        for line in open(txt_path).read().strip().splitlines():
            c, b = yolo_line_to_xyxy(line, W, H)
            cv2.rectangle(out, (int(b[0]), int(b[1])), (int(b[2]), int(b[3])), COLORS.get(c, (255, 255, 255)), 2)
    return out

names = sorted(os.path.splitext(f)[0] for f in os.listdir(IMG_DIR))
for name in names[:: max(1, len(names) // 4)][:4]:
    img = cv2.imread(os.path.join(IMG_DIR, name + ".jpg"))
    plt.figure(figsize=(16, 9))
    plt.imshow(cv2.cvtColor(draw_label_file(img, os.path.join(PRE_DIR, name + ".txt")), cv2.COLOR_BGR2RGB))
    plt.title(name); plt.axis("off"); plt.show()

In [ ]:
# Roboflow 업로드용 zip (YOLO 형식: images/ + labels/ + data.yaml)
UPLOAD_ZIP = os.path.join(EVAL_DIR, "eval_prelabels_yolo.zip")
with zipfile.ZipFile(UPLOAD_ZIP, "w") as z:
    for name in names:
        z.write(os.path.join(IMG_DIR, name + ".jpg"), f"images/{name}.jpg")
        z.write(os.path.join(PRE_DIR, name + ".txt"), f"labels/{name}.txt")
    z.writestr("data.yaml", "names:\n" + "".join(f"  - {n}\n" for n in CLASS_NAMES) + f"nc: {len(CLASS_NAMES)}\n")
print("✅ 업로드용 zip:", UPLOAD_ZIP, f"({len(names)}장)")

from google.colab import files
files.download(UPLOAD_ZIP)

## 🏷️ Roboflow 라벨링 가이드

**1. 업로드**: Roboflow에서 새 프로젝트(Object Detection) → 위 zip을 통째로 드래그 → 박스가 그려진 채로 올라옵니다.

**2. 수정 규칙** (이 기준만 지키면 됩니다)

| 상황 | 할 일 |
|---|---|
| 그림자·잔디·장비를 잡은 박스 | **삭제** |
| 박스 없는 선수/심판 | **추가** (대략 감싸기만, 정밀할 필요 없음) |
| 회색 `Unknown` 박스 | **Hongik / Kookmin / Referee 중 하나로 바꾸거나 삭제** ← 정답에는 Unknown이 남으면 안 됨 |
| 클래스가 틀린 박스 | **수정** |
| 한 사람에 박스 2개 | 하나 **삭제** |
| 두 사람을 감싼 박스 1개 | 하나로 줄이고 나머지 한 명 **추가** |
| 경기장(초록 경계) 밖 사람 | **라벨링하지 않음** |
| 반쯤 가려진 선수 | **보이는 만큼 박스** |

**3. 내보내기**: Generate Version (⚠️ Preprocessing의 Resize는 끄고, Augmentation 없음) → Export → **YOLOv8** 형식 zip 다운로드

**4.** 받은 zip을 Drive의 `pipeline_outputs/eval/` 폴더에 **`gt_roboflow.zip`** 이름으로 넣고 파트 B를 실행하세요.

---
# 파트 B. 평가

런타임이 새로 시작됐다면 **공통 설정 3개 셀**만 다시 실행한 뒤 여기서부터 실행하세요. (파트 A는 다시 돌릴 필요 없음)

In [ ]:
GT_ZIP = os.path.join(EVAL_DIR, "gt_roboflow.zip")
GT_DIR = os.path.join("/content", "gt_extracted")
shutil.rmtree(GT_DIR, ignore_errors=True)
with zipfile.ZipFile(GT_ZIP) as z:
    z.extractall(GT_DIR)


def original_name(fname):
    """Roboflow가 바꾼 파일명 → 원래 이름. 예: 'DJI_..._f00413_jpg.rf.ab12cd.txt' → 'DJI_..._f00413'"""
    base = os.path.basename(fname)
    base = re.split(r"_(?:jpg|jpeg|png)\.rf\.", base, flags=re.IGNORECASE)[0]
    return os.path.splitext(base)[0]


def read_names_yaml(gt_dir):
    """Roboflow data.yaml의 클래스 순서를 읽음 (Roboflow가 순서를 바꿀 수 있어서 이름으로 다시 매핑)"""
    ymls = glob.glob(os.path.join(gt_dir, "**", "data.yaml"), recursive=True)
    if not ymls:
        return CLASS_NAMES
    txt = open(ymls[0], encoding="utf-8").read()
    m = re.search(r"names:\s*\[(.*?)\]", txt, re.S)
    if m:
        return [s.strip().strip("'\"") for s in m.group(1).split(",") if s.strip()]
    return re.findall(r"^\s*-\s*['\"]?([^'\"\n]+?)['\"]?\s*$", txt.split("names:")[1], re.M)


def load_gt(gt_dir):
    """→ {name: {"W","H","boxes":[(class_name, xyxy), ...]}}"""
    rf_names = read_names_yaml(gt_dir)
    gt = {}
    for lbl in glob.glob(os.path.join(gt_dir, "**", "labels", "*.txt"), recursive=True):
        name = original_name(lbl)
        img = cv2.imread(os.path.join(IMG_DIR, name + ".jpg"))   # 원본 해상도 기준으로 복원
        if img is None:
            print("⚠️ 원본 이미지를 찾을 수 없음:", name); continue
        H, W = img.shape[:2]
        boxes = []
        for line in open(lbl).read().strip().splitlines():
            if not line.strip():
                continue
            c, b = yolo_line_to_xyxy(line, W, H)
            boxes.append((rf_names[c], b))
        gt[name] = {"W": W, "H": H, "boxes": boxes}
    return gt


GT = load_gt(GT_DIR)
_cnt = Counter(c for v in GT.values() for c, _ in v["boxes"])
print(f"정답 이미지 {len(GT)}장, 박스 {sum(_cnt.values())}개 → {dict(_cnt)}")
if _cnt.get("Unknown"):
    print(f"⚠️ 정답에 Unknown이 {_cnt['Unknown']}개 남아 있습니다. 탐지 평가에는 포함되지만 팀 정확도에서는 제외됩니다.")

In [ ]:
EVAL_TEAM = ["Hongik", "Kookmin", "Referee"]


def detection_and_team_metrics(run_json_by_stem):
    """정답 vs 예측: 탐지(클래스 무관) P/R/F1 + 짝지어진 박스의 클래스 정확도 + 혼동행렬"""
    tp = fp = fn = 0
    conf_mat = Counter()                     # (정답 클래스, 예측 클래스)
    per_frame = []
    for stem, jpath in run_json_by_stem.items():
        by_frame, _ = load_tracking(jpath)
        for name, g in GT.items():
            if not name.startswith(stem):
                continue
            fid = int(name.split("_f")[-1])
            preds = by_frame.get(fid, [])
            gb = [b for _, b in g["boxes"]]
            pb = [p["bbox"] for p in preds]
            pairs, ug, up = match_by_center(gb, pb)
            tp += len(pairs); fn += len(ug); fp += len(up)
            for gi, pi in pairs:
                conf_mat[(g["boxes"][gi][0], preds[pi]["type"])] += 1
            per_frame.append({"image": name, "gt": len(gb), "pred": len(pb), "TP": len(pairs), "FN": len(ug), "FP": len(up)})
    P = tp / (tp + fp) if tp + fp else 0.0
    R = tp / (tp + fn) if tp + fn else 0.0
    F1 = 2 * P * R / (P + R) if P + R else 0.0
    team_pairs = {k: v for k, v in conf_mat.items() if k[0] in EVAL_TEAM}
    correct = sum(v for (g, p), v in team_pairs.items() if g == p)
    team_acc = correct / sum(team_pairs.values()) if team_pairs else 0.0
    ref_gt = sum(1 for v in GT.values() for c, _ in v["boxes"] if c == "Referee")
    ref_hit = conf_mat.get(("Referee", "Referee"), 0)
    return {
        "det_precision": round(P, 3), "det_recall": round(R, 3), "det_f1": round(F1, 3),
        "TP": tp, "FP": fp, "FN": fn,
        "team_acc": round(team_acc, 3),
        "referee_recall": round(ref_hit / ref_gt, 3) if ref_gt else None,
    }, conf_mat, pd.DataFrame(per_frame)


def scorer_track_metrics(stem, run_json, cfg, verbose=True):
    """
    득점 선수 추적 품질 (라벨링 없이 측정).
    - 정답 위치: v3에서 사람이 확인해 이어붙인 득점 선수의 발 위치 (cfg["scorer"])
    - 평가할 결과(run_json)에서 매 프레임 그 위치에 가장 가까운 박스의 track_id를 읽음

    지표
    - scorer_ids / scorer_id_switches : '정답 위치가 확실한 프레임'에서만 계산 (믿을 수 있는 값)
    - scorer_coverage : v3가 놓친 구간은 앞뒤 위치를 직선으로 이어 정답 위치를 '추정'해서 포함
                        → 그 구간에서는 근처의 다른 선수가 잡힐 수 있으니 참고용(근사값)
    """
    ref_by_frame, _ = load_tracking(tracking_json_of(stem))       # 정답 경로는 항상 v3 기준
    run_by_frame, _ = load_tracking(run_json)
    known = {}
    for tid, (a, b) in cfg["scorer"]:
        for fid in range(a, b + 1):
            for p in ref_by_frame.get(fid, []):
                if p["track_id"] == tid:
                    known[fid] = p["feet"]
    if not known:
        print(f"⚠️ {stem}: v3 JSON에서 득점 선수 track_id를 찾지 못했습니다. EVAL_CLIPS의 scorer 값을 확인하세요.")
        return {"scorer_frames": 0, "scorer_coverage": 0.0, "scorer_ids": 0, "scorer_id_switches": 0}
    span = range(min(known), max(known) + 1)
    kf = np.array(sorted(known))
    kx = np.array([known[f][0] for f in kf]); ky = np.array([known[f][1] for f in kf])

    seq = []   # (frame, 찾은 track_id 또는 None, 정답 위치가 확실한지)
    for fid in span:
        gx, gy = (known[fid] if fid in known else (np.interp(fid, kf, kx), np.interp(fid, kf, ky)))
        best, bd = None, 1e9
        for p in run_by_frame.get(fid, []):
            d = np.hypot(p["feet"][0] - gx, p["feet"][1] - gy)
            if d < bd:
                best, bd = p, d
        tid = best["track_id"] if best is not None and bd <= SCORER_MATCH_PX else None
        seq.append((fid, tid, fid in known))

    sure = [t for _, t, k in seq if k and t is not None]
    switches = sum(1 for a, b in zip(sure, sure[1:]) if a != b)
    covered = sum(1 for _, t, _ in seq if t is not None)

    if verbose:   # 어느 ID가 어느 구간을 맡았는지 (v3 노트북의 SEGMENTS와 같은 형태) → 영상으로 확인할 때 사용
        runs, cur = [], None
        for fid, t, _ in seq:
            if cur and cur[0] == t and cur[2] == fid - 1:
                cur[2] = fid
            else:
                cur = [t, fid, fid]; runs.append(cur)
        desc = ", ".join(f"{'없음' if t is None else '#' + str(t)}({a}~{b})" for t, a, b in runs)
        print(f"  [{stem}] 득점 선수 경로: {desc}")

    return {
        "scorer_frames": len(seq),
        "scorer_coverage": round(covered / len(seq), 3),
        "scorer_ids": len(set(sure)),
        "scorer_id_switches": switches,
    }


def segment_stats(stem, run_json, cfg):
    """플레이 구간 안의 고유 ID 수, Unknown 비율 (라벨링 없이 측정)"""
    by_frame, _ = load_tracking(run_json)
    a, b = cfg["play_range"]
    ids, types = set(), Counter()
    for fid in range(a, b + 1):
        for p in by_frame.get(fid, []):
            ids.add(p["track_id"]); types[p["type"]] += 1
    total = sum(types.values())
    return {"segment_unique_ids": len(ids),
            "segment_unknown_ratio": round(types["Unknown"] / total, 3) if total else 0.0}


RESULTS_CSV = os.path.join(EVAL_DIR, "eval_results.csv")


def evaluate_run(run_name, run_json_by_stem, note=""):
    """
    한 번의 실험 결과를 채점해서 eval_results.csv에 한 줄 추가합니다.
    run_json_by_stem: {stem: 그 실험이 만든 tracking_final.json 경로}
    """
    det, conf_mat, per_frame = detection_and_team_metrics(run_json_by_stem)
    row = {"run": run_name, "note": note, **det}
    for stem, jpath in run_json_by_stem.items():
        cfg = EVAL_CLIPS[stem]
        k = cfg["clip_key"]
        for key, val in {**scorer_track_metrics(stem, jpath, cfg), **segment_stats(stem, jpath, cfg)}.items():
            row[f"{key}_{k}"] = val
    df_new = pd.DataFrame([row])
    if os.path.exists(RESULTS_CSV):
        old = pd.read_csv(RESULTS_CSV)
        old = old[old["run"] != run_name]                      # 같은 이름으로 다시 돌리면 덮어쓰기
        df_new = pd.concat([old, df_new], ignore_index=True)
    df_new.to_csv(RESULTS_CSV, index=False, encoding="utf-8-sig")
    return row, conf_mat, per_frame


def show_confusion(conf_mat):
    rows = EVAL_TEAM + (["Unknown"] if any(g == "Unknown" for g, _ in conf_mat) else [])
    cols = CLASS_NAMES
    return pd.DataFrame([[conf_mat.get((r, c), 0) for c in cols] for r in rows],
                        index=[f"정답:{r}" for r in rows], columns=[f"예측:{c}" for c in cols])

print("✅ 평가 함수 준비 완료")

In [ ]:
# ===== v3 Before 측정 =====
V3_RUN = {stem: tracking_json_of(stem) for stem in EVAL_CLIPS}
row, conf_mat, per_frame = evaluate_run("v3_baseline", V3_RUN, note="yolov8x imgsz960 conf0.15 + custom ByteTrack + RF 분류기")

print("=== v3 Before ===")
for k, v in row.items():
    print(f"{k:>28}: {v}")
print("\n=== 팀 분류 혼동행렬 (짝지어진 박스만) ===")
display(show_confusion(conf_mat))
print("\n=== 프레임별 ===")
display(per_frame)

In [ ]:
# 틀린 곳을 눈으로 확인: 초록=맞게 찾음(TP), 빨강=놓친 정답(FN), 파랑=잘못 잡음(FP)
def show_errors(name, run_json_by_stem):
    stem = name.split("_f")[0]
    fid = int(name.split("_f")[-1])
    by_frame, poly = load_tracking(run_json_by_stem[stem])
    preds = by_frame.get(fid, [])
    g = GT[name]
    pairs, ug, up = match_by_center([b for _, b in g["boxes"]], [p["bbox"] for p in preds])
    img = cv2.imread(os.path.join(IMG_DIR, name + ".jpg"))
    cv2.polylines(img, [poly.reshape(-1, 1, 2)], True, (0, 255, 0), 1)
    for gi, pi in pairs:
        b = preds[pi]["bbox"]; cv2.rectangle(img, (int(b[0]), int(b[1])), (int(b[2]), int(b[3])), (0, 200, 0), 2)
    for gi in ug:
        b = g["boxes"][gi][1]; cv2.rectangle(img, (int(b[0]), int(b[1])), (int(b[2]), int(b[3])), (0, 0, 255), 2)
    for pi in up:
        b = preds[pi]["bbox"]; cv2.rectangle(img, (int(b[0]), int(b[1])), (int(b[2]), int(b[3])), (255, 0, 0), 2)
    plt.figure(figsize=(16, 9)); plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(f"{name}  TP={len(pairs)}  FN={len(ug)}  FP={len(up)}"); plt.axis("off"); plt.show()

# 놓친 선수가 가장 많은 프레임 2장
for name in per_frame.sort_values("FN", ascending=False)["image"].head(2):
    show_errors(name, V3_RUN)

## 다음 단계에서 쓰는 법

1단계에서 탐지 설정을 바꿔 새 JSON을 만들었다면 (v3와 같은 형식):

```python
NEW_RUN = {stem: "/content/.../새결과_tracking_final.json" for stem in EVAL_CLIPS}
row, conf_mat, per_frame = evaluate_run("step1a_imgsz1920", NEW_RUN, note="imgsz 960→1920")
display(pd.read_csv(RESULTS_CSV))   # 지금까지의 모든 실험 비교표
```

`eval_results.csv`의 한 줄 한 줄이 자소서의 Before/After 표가 됩니다.